# Twinned FCC 3D Pipeline: Advanced Walkthrough (Optimization)

A full run of the Twinned FCC pipeline, built from step functions (`steps/`) that you can inspect and reuse outside this notebook. On top of the standard pipeline it adds two things:

- **Part K (Orientation Assignment)** tries several assignment mode/seed combinations, scores each against a real EBSD pole-figure reference, and keeps the best one. A plain run would do a single unscored assignment.
- **A temporal-slice sweep** at the end re-runs the downstream stages (host allocation through cleaning) for every saved Monte Carlo slice you ask for, and tells you which ones reach a given percentage of the target twin volume fraction.

**Pipeline stages:** EBSD analysis, Monte Carlo base grain structure, representativeness assessment, host allocation, orientation assignment (with optimization), twin generation, post-twin validation, distribution comparison, subsetting, visualization/export, and the temporal-slice sweep. Voronoi and image-import base structures aren't covered; Monte Carlo is used throughout.

The default parameters were validated on the reference dataset used here.

## Part A: Start (Global Configuration)

Sets the output directory and opens the report session (`upxo.reporting.ReportSession`) that later stages add results to. Everything this notebook produces (raw exports, Abaqus mesh files, the report) goes under `<output_dir>/TwinnedFCC/...`.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

from upxo.pxtal.twinned_simple_3d.steps import (
    steps_start, steps_ebsd_analysis_1, steps_self_repr_1, steps_ebsd_analysis_2,
    steps_base_grain_structure_mc, steps_repr_assessment_qualification,
    steps_transformations, steps_host_allocation, steps_pre_twin_validation,
    steps_orientation_assignment, steps_twin_generation, steps_post_twin_validation,
    steps_distribution_viewer, steps_subsetting, steps_visualization_export,
    steps_temporal_slice_sweep,
)

In [ ]:
RESEARCHER_NAME = ""
ORGANISATION = ""
EMAIL = ""
MATERIAL_NAME = "OFHC Copper"
PROCESSING_CONDITION = ""

report = steps_start.open_pipeline_report(
    researcher_name=RESEARCHER_NAME, organisation=ORGANISATION, email=EMAIL,
    material_name=MATERIAL_NAME, processing_condition=PROCESSING_CONDITION,
    notebook_tag='adv1')
report

## Part B: EBSD Analysis-1

Loads the reference EBSD map, detects grains and trims a one-percent border to avoid edge artefacts. The map is subsampled by default, because grain detection on a full-resolution map is slow. The result is the `repgen2d` object (`rg`) used for the rest of the pipeline.

In [ ]:
CTF_FILE = r'C:\Development\EBSD datasets\UKAEA__OFHCCu\OFHC_Cu_dataset\EBSD_pre\warp_out_s2.ctf'
STRIDE_X, STRIDE_Y = 5, 5   # default subsampling

rdr = steps_ebsd_analysis_1.subsample_and_load(
    CTF_FILE, subsample=True, stride_x=STRIDE_X, stride_y=STRIDE_Y, reuse_subsampled=True)

In [ ]:
MIN_GRAIN_SIZE_DETECT = 10   # pixels
MISORI_TOL = 10.0           # degrees
rdr = steps_ebsd_analysis_1.detect_grains(rdr, min_grain_size=MIN_GRAIN_SIZE_DETECT, misori_tol=MISORI_TOL)

In [ ]:
rdr = steps_ebsd_analysis_1.crop(rdr, xstart_pct=1.0, ystart_pct=1.0, xend_pct=99.0, yend_pct=99.0)

In [ ]:
rg = steps_ebsd_analysis_1.clean_and_characterize(rdr, ctf_file=CTF_FILE)

## Part C: Self-Representativeness-1 (Optional)

Off by default. Checks whether a smaller moving-rectangle sub-domain reproduces the grain morphology of the whole parent domain, which tells you how far to trust the EBSD map as a statistical reference. The main pipeline doesn't need it, so skip to Part D if you don't care.

In [ ]:
boundary_stats = steps_self_repr_1.boundary_grain_ratio(rg)
boundary_stats

In [ ]:
subsets = steps_self_repr_1.extract_subsets(rg, start_pct=0.0, length_pct=20.0, stride_pct=20.0)
morph = steps_self_repr_1.compute_morphology(rg, subsets, params=('area', 'perimeter'))
assessment = steps_self_repr_1.assess_representativeness(morph, metrics=('ks_sim', 'wasserstein_sim'))
len(subsets), len(assessment)

## Part D: EBSD Analysis-2

The heaviest analysis stage. It computes the misorientation distribution function (MDF), finds candidate coincidence-site-lattice (CSL) peaks, splits parent/twin grain pairs around the Sigma3 (twin) peak, classifies the twin role of every grain, splits the twin volume fraction into Stage-1, Stage-2a and Stage-2b targets, and measures twin lamella thickness. The outputs (`parent_info`, `vf_targets`, `twin_thickness`) feed almost every later stage.

In [ ]:
mdf, peaks = steps_ebsd_analysis_2.compute_mdf_and_peaks(rg)
selected_peaks = steps_ebsd_analysis_2.select_all_peaks(peaks)   # keep every detected peak
len(selected_peaks['indices'])

In [ ]:
csl_grains = steps_ebsd_analysis_2.segregate_csl_pairs(rg, mdf, peaks, selected_peaks)
parent_info = steps_ebsd_analysis_2.identify_parent_grains(rg, csl_grains)

In [ ]:
CSL_LABEL = steps_ebsd_analysis_2.DEFAULT_CSL_LABEL   # 'S3  (twin)'
tvf_by_label = steps_ebsd_analysis_2.compute_twin_area_fractions(rg, parent_info, csl_labels=(CSL_LABEL,))
tvf_by_label[CSL_LABEL]

In [ ]:
role_props = steps_ebsd_analysis_2.compute_grain_role_properties(rg, parent_info, selected_props=('area',))
vf_partition, vf_targets = steps_ebsd_analysis_2.compute_vf_partition(rg, parent_info, tvf_by_label, csl_label=CSL_LABEL)
vf_targets

In [ ]:
TWIN_THICKNESS_ABRUPT_THRESHOLD = 0.8   # twin-extent/parent-extent ratio below which a twin counts as abruptly-ending
twin_thickness = steps_ebsd_analysis_2.compute_twin_thickness(
    rg, parent_info, abrupt_threshold=TWIN_THICKNESS_ABRUPT_THRESHOLD)
twin_thickness['mean'], twin_thickness['median']

In [ ]:
report.add_table([vf_targets], title='EBSD VF Targets', section='EBSD Analysis-2')
twin_thickness_summary = {k: v for k, v in twin_thickness.items() if not hasattr(v, '__len__') or isinstance(v, str)}
report.add_table([twin_thickness_summary], title='EBSD Twin Thickness Statistics', section='EBSD Analysis-2')

### Pole Figures

Pole figures of the EBSD reference for four populations: the full map, parents only, twins only, and parents and twins overlaid with different markers. Defaults: {111} poles, scatter plots, stereographic projection of the upper hemisphere, with sample symmetry applied (every orientation is copied through 180 degree rotations about RD, TD and ND). Cubic crystal symmetry is always applied through the symmetric equivalents of the pole family. `POLE_FIGURE_PLOT_MODE` picks `'scatter'`, `'density'` (multiples-of-uniform-density contours) or `'hybrid'`; `POLE_FIGURE_HALF_WIDTH_DEG` is the angular half-width of the density kernel. These settings apply to every pole figure in the notebook.

In [ ]:
POLE_FIGURE_HALF_WIDTH_DEG = 7.5
POLE_FIGURE_FAMILY = '111'             # '100', '110' or '111'
POLE_FIGURE_PLOT_MODE = 'scatter'      # 'density' (MUD) or 'hybrid'
PF_APPLY_SAMPLE_SYMMETRY = True        # 180 deg rotations about RD, TD and ND
PF_USE_RD = True
PF_USE_TD = True
PF_USE_ND = True
PF_PROJECTION = 'stereographic'        # or 'equal_area'
PF_HEMISPHERE = 'upper'                # 'lower' or 'both_mapped' (scatter and hybrid)
PF_VIEW = dict(apply_sample_symmetry=PF_APPLY_SAMPLE_SYMMETRY, use_rd=PF_USE_RD, use_td=PF_USE_TD, use_nd=PF_USE_ND,
               projection=PF_PROJECTION, hemisphere=PF_HEMISPHERE)
PF_OVERLAY_VIEW = PF_VIEW

ebsd_pf_stages = steps_ebsd_analysis_2.ebsd_pole_figure_stages(rg, parent_info, csl_label=CSL_LABEL)

fig, ax = steps_visualization_export.plot_pole_figure(
    ebsd_pf_stages['full'], pole_family=POLE_FIGURE_FAMILY, plot_mode=POLE_FIGURE_PLOT_MODE,
    half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG, **PF_VIEW, title='EBSD Full Structure')
report.add_image(fig, title='EBSD Pole Figure -- Full Structure', section='EBSD Analysis-2')
plt.show()

fig, ax = steps_visualization_export.plot_pole_figure(
    ebsd_pf_stages['parents'], pole_family=POLE_FIGURE_FAMILY, plot_mode=POLE_FIGURE_PLOT_MODE,
    half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG, **PF_VIEW, title='EBSD Parents Only')
report.add_image(fig, title='EBSD Pole Figure -- Parents Only', section='EBSD Analysis-2')
plt.show()

fig, ax = steps_visualization_export.plot_pole_figure(
    ebsd_pf_stages['all_twins'], pole_family=POLE_FIGURE_FAMILY, plot_mode=POLE_FIGURE_PLOT_MODE,
    half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG, **PF_VIEW, title='EBSD Twins Only')
report.add_image(fig, title='EBSD Pole Figure -- Twins Only', section='EBSD Analysis-2')
plt.show()

fig, ax = steps_visualization_export.plot_pole_figure_overlay(
    ebsd_pf_stages['parents'], 'Parents', ebsd_pf_stages['all_twins'], 'Twins',
    pole_family=POLE_FIGURE_FAMILY, **PF_OVERLAY_VIEW, title='EBSD Parents vs. Twins')
report.add_image(fig, title='EBSD Pole Figure -- Parents vs Twins Overlay', section='EBSD Analysis-2')
plt.show()

## Part E: Self-Representativeness-3 (Omitted)

Texture and pole-figure analysis for this stage isn't implemented in the package yet, so there's nothing to wrap. The `twinned_simple_3d` docs advise against placeholder implementations for production studies, so there's no `steps_self_repr_3.py` module and no cell for it here.

## Part F: Base Grain Structure (Monte Carlo)

Monte Carlo is the only base-structure method used in this notebook (Voronoi and image import aren't covered). This stage runs a 3D Potts-model Monte Carlo grain-growth simulation, computes the labelled feature index (grain labels) for every stored temporal slice, and cleans each slice by merging undersized grains and removing single-voxel spikes.

In [ ]:
# NOTE: a full 50x50x50, 100-step run can take a while --
# shrink XMAX/YMAX/ZMAX/MCSTEPS for a quick smoke-test run.
pxt = steps_base_grain_structure_mc.run_mc_simulation(
    xmax=50.0, ymax=50.0, zmax=50.0, q_states=10, mcsteps=100, save_interval=5,
    mcalg='300b', consider_boltzmann=True, boltzmann_temp_factor=0.01, rng_seed=0)

In [ ]:
lfi = steps_base_grain_structure_mc.calculate_lfi(pxt)

In [ ]:
cumulative_summary, n_passes_run = steps_base_grain_structure_mc.clean_structure(pxt, min_grain_size=4, start_index=1)
n_passes_run

## Part G: Representativeness Assessment and Qualification

Ranks every stored Monte Carlo temporal slice against the EBSD parent-grain reference (2D cross-sections compared with the EBSD map), re-checks the candidates against the tolerances you set, and picks the single best match as the structure used for host allocation. It also calibrates the physical scale (um/voxel) of the chosen slice.

Ranking starts at slice 1, not 0. Slice 0 is the un-annealed seed state: as many grains as voxels, no growth. It isn't a meaningful candidate, but its huge grain count can dominate the shortlist under some criteria even though it's the worst match. The Base Grain Structure cleaning step skips it for the same reason (`start_index=1`).

`QUAL_NG_TOLERANCE_PCT` and `QUAL_SCALE_CHECK_TOLERANCE_PCT` set how strict the shortlist star-rating is. Tighter tolerances mean fewer candidates get full marks, and that can change which slice is selected.

In [ ]:
QUAL_N_COMPARISON_SLICES = 5          # 2D cross-sections sampled per axis, per candidate slice
QUAL_SCALE_CHECK_TOLERANCE_PCT = 5.0  # allowed drift (%) in the calibrated scale factor across candidates
QUAL_NG_TOLERANCE_PCT = 5.0           # allowed relative deviation (%) in grain count vs. EBSD before a candidate loses a star

candidates = steps_repr_assessment_qualification.rank_slices(
    pxt, parent_info=parent_info, role_props=role_props, start=1,
    n_comparison_slices=QUAL_N_COMPARISON_SLICES,
    scale_check_tolerance_pct=QUAL_SCALE_CHECK_TOLERANCE_PCT)
len(candidates)

In [ ]:
candidates = steps_repr_assessment_qualification.reassess_candidates(
    pxt, candidates, role_props, scale_check_tolerance_pct=QUAL_SCALE_CHECK_TOLERANCE_PCT)

In [ ]:
QUAL_SHORTLIST_CONFIG = {
    'MC_NG_TOLERANCE_PCT': QUAL_NG_TOLERANCE_PCT,
    'MC_PROPERTY_TOLERANCE_PCT_PER_PROP': {},
    'MC_SCALE_CHECK_TOLERANCE_PCT': QUAL_SCALE_CHECK_TOLERANCE_PCT,
}
rows, best_row = steps_repr_assessment_qualification.shortlist_and_select(
    candidates, role_props, config=QUAL_SHORTLIST_CONFIG)
best_row['tslice_key'] if best_row else None

In [ ]:
scale_factor = steps_repr_assessment_qualification.apply_selected_scale_factor(pxt, best_row)
tslice_key = best_row['tslice_key']
scale_factor

In [ ]:
report.add_table([rows[0]] if rows else [{}], title='Temporal Slice Shortlist', section='Repr. assess. & qualification')

## Part H: Transformations (Optional)

All scaling factors default to 1.0, i.e. nothing changes. You can skip this cell entirely and the selected slice goes on unmodified. If you run it with other factors, the structure is rescaled and stretched (so grains are no longer equiaxed) before host allocation.

In [ ]:
SCALE_FACTOR, SF_X, SF_Y, SF_Z = 1.0, 1.0, 1.0, 1.0   # identity -- no transform
transformed_base, original_base = steps_transformations.apply_transform(
    pxt, tslice_key, scale_factor=SCALE_FACTOR, sf_x=SF_X, sf_y=SF_Y, sf_z=SF_Z)

## Part I: Host Allocation

Decides which grains in the base structure become twin "hosts". It uses a spatial selection constrained to a maximal independent set, aimed at a target volume fraction, and then re-checks the result on 2D cross-sections against the EBSD 2D hosting-fraction target.

`HOST_TARGET_FRACTION` is a fraction of grain *count*, taken from the EBSD `twin_hosting_fraction` (the fraction of parent grains that host a twin). It is not an area or volume fraction. The achieved hosting fraction reported below is a volume fraction, so the two can't be compared directly. How much volume a given grain-count selection captures depends a lot on `HOST_RANKING_VOLUME_WEIGHT` and `HOST_POOL_B_BAND_SHAPE`. Here, ranking purely by volume (1.0) and drawing Pool B from the large-grain tail (`'above'`) favours big grains, so with a right-skewed grain-size distribution even a modest grain-count fraction can grab a disproportionate share of the total volume.

In [ ]:
HOST_TARGET_FRACTION = 0.3          # EBSD 2D hosting fraction (grain-count based)
HOST_MIN_VOXELS = 4                 # grains smaller than this are never host-eligible
HOST_2D_TO_3D_SCALE_FACTOR = 1.65   # corrects 2D EBSD under-sampling of the true 3D hosting fraction
HOST_RANKING_VOLUME_WEIGHT = 1.0    # 0=rank by adjacency count, 1=rank by volume (largest first), 0.5=equal blend
HOST_MIS_FRACTION = 0.25            # fraction of hosts drawn from the non-adjacent (MIS) pool
HOST_MIS_RUNS = 10                  # number of random MIS trials; the largest is kept
HOST_SEED = 0
HOST_POOL_B_BAND_SHAPE = 'above'    # 'below'/'between'/'above' -- which size-tail Pool B draws from
HOST_POOL_B_N_STD = 1.0             # Pool B band half-width, in std devs of grain size

base = steps_host_allocation.allocate_hosts(
    pxt, tslice_key, transformed_base=transformed_base,   # None to use the raw MC slice instead
    target_fraction=HOST_TARGET_FRACTION, min_voxels=HOST_MIN_VOXELS,
    scale_factor=HOST_2D_TO_3D_SCALE_FACTOR, ranking_weight=HOST_RANKING_VOLUME_WEIGHT,
    mis_fraction=HOST_MIS_FRACTION, mis_runs=HOST_MIS_RUNS, seed=HOST_SEED,
    pool_b_band_shape=HOST_POOL_B_BAND_SHAPE, pool_b_n_std=HOST_POOL_B_N_STD)

In [ ]:
hosting_2d = steps_host_allocation.reassess_hosting_2d(base)
hosting_2d

In [ ]:
report.add_table([hosting_2d], title='Host Allocation 2D Re-Assessment', section='Host Allocation')

### Grain Structure Summary

Overall grain-count and volume statistics for the allocated base structure, plus a host versus non-host volume breakdown. It gives some context for the hosting fraction targeted above.

In [ ]:
vols = np.array(list(base.mprop['volnv'].values()))
host_vols = np.array([base.mprop['volnv'][g] for g in base.host_grain_ids])
nonhost_vols = np.array([base.mprop['volnv'][g] for g in base.non_host_grain_ids])

print(f'Base structure: {len(vols)} grains total')
print(f'  Volume (voxels)  mean={vols.mean():.1f}  median={np.median(vols):.1f}  '
      f'min={vols.min():.0f}  max={vols.max():.0f}')
print()
print(f'Host grains:     {len(host_vols)}  '
      f'mean_vol={host_vols.mean():.1f} vox  median_vol={np.median(host_vols):.1f} vox')
print(f'Non-host grains: {len(nonhost_vols)}  '
      f'mean_vol={nonhost_vols.mean():.1f} vox  median_vol={np.median(nonhost_vols):.1f} vox')

## Part J: Pre-Twin Validation

Checks the grain-size distribution of the host-allocated base structure against the EBSD reference with twins merged into their parent grains (the base structure has no twins yet). It's done per axis, on 2D cross-sections, with a Wasserstein-distance pass threshold.

`PRETWIN_P_PERCENTAGE` is the minimum percentage of sampled slices, per axis, that must pass the Wasserstein criterion for that axis to be accepted. `PRETWIN_WASSERSTEIN_THRESHOLD` is the largest Wasserstein distance between one slice's grain-size distribution and the EBSD reference for that slice to count as a pass.

In [ ]:
PRETWIN_N_SLICES = (10, 10, 10)              # 2D cross-sections sampled per axis (x, y, z)
PRETWIN_P_PERCENTAGE = (60.0, 60.0, 60.0)    # required per-axis pass rate (%) for that axis to be accepted
PRETWIN_WASSERSTEIN_THRESHOLD = 0.5          # max Wasserstein distance for one slice to count as passing

pre_validator = steps_pre_twin_validation.validate_morphological_representativeness(
    base, rg, parent_info, n_slices=PRETWIN_N_SLICES, test_axes=('x', 'y', 'z'),
    p_percentage=PRETWIN_P_PERCENTAGE, wasserstein_threshold=PRETWIN_WASSERSTEIN_THRESHOLD)
pre_validator.overall_accepted

In [ ]:
print(pre_validator.report())

### Per-Axis Pass Rate

The fraction of tested slices, per axis, that pass the Wasserstein-distance criterion.

In [ ]:
ax_names = list(pre_validator.axis_acceptance.keys())
pct_pass = [pre_validator.axis_acceptance[a]['pct_pass'] for a in ax_names]
colors = ['seagreen' if pre_validator.axis_acceptance[a]['accepted'] else 'crimson' for a in ax_names]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(ax_names, pct_pass, color=colors, alpha=0.75, edgecolor='black')
ax.axhline(60.0, color='steelblue', linestyle='--', linewidth=2, label='Pass threshold (60%)')
ax.set_ylabel('Slices passing (%)')
ax.set_title('Pre-Twin morphological representativeness -- per-axis pass rate')
ax.set_ylim(0, 105)
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar, pct in zip(bars, pct_pass):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{pct:.0f}%',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

## Part K: Orientation Assignment

Tries several orientation-assignment mode/seed combinations and scores each one against the real EBSD "pure parents" pole-figure reference. The best match is kept, instead of a single unscored `conflict_free` run.

`OPT_SELECTED` says which modes to sweep and how many iterations each (an iterable of `(mode, n_iterations)` pairs). `mrf_gibbs` and `mrf_map` exist but are much slower, so they're left out of the default selection. Each run's synthetic host-grain pole figure is compared cell by cell against the EBSD reference (the CSL-`csl_label` pure-parent grains, which in the EBSD data always host this twin type and are never twins themselves). Runs are ranked by the interquartile range (IQR) of that cell-by-cell difference, smallest first, so a smaller IQR means a tighter match across the whole pole figure. `OPT_UNIT_NORMALIZE = True` (the default) compares pattern shape and ignores absolute texture strength.

In [ ]:
OPT_SELECTED = (('conflict_free', 5), ('paired_pool', 5))   # (mode, n_iterations) pairs to sweep
OPT_BASE_SEED = 42
OPT_POLE_FAMILY = '100'
OPT_RESOLUTION = 'Medium'          # 'Low'/'Medium'/'High' pole-figure grid resolution
OPT_UNIT_NORMALIZE = True

records, top10 = steps_orientation_assignment.optimize_mapping(
    base, rg, parent_info, csl_label=CSL_LABEL, selected=OPT_SELECTED,
    base_seed=OPT_BASE_SEED, pole_family=OPT_POLE_FAMILY, resolution=OPT_RESOLUTION,
    unit_normalize=OPT_UNIT_NORMALIZE)
print(f'{len(records)} successful run(s); {len(top10)} lowest-IQR run(s) retained for inspection.')

In [ ]:
fig, ax = steps_orientation_assignment.plot_optimize_mapping_distribution(records)
report.add_image(fig, title='Optimize Mapping -- IQR Distribution by Mode', section='Orientation Assignment')
plt.show()

In [ ]:
assigner = top10[0]['assigner']
print(f"Selected: mode={top10[0]['mode']}  seed={top10[0]['seed']}  "
      f"IQR={top10[0]['iqr']:.4f}  n_conflicts={assigner.n_conflicts}")

optimize_table_rows = [{k: v for k, v in cand.items() if k != 'assigner'} for cand in top10]
report.add_table(optimize_table_rows, title='Optimize Mapping -- Top Candidates', section='Orientation Assignment')

### Pole Figure Comparison: EBSD versus Pre-Twin Synthetic

EBSD "pure parents" of the selected CSL type (grains that host its twins and are never twins themselves) against the synthetic twin-host grains, before twin generation. If the textures don't match, this tells you whether the problem starts in orientation assignment. `PFCMP_PLOT_MODE` is `'scatter'` (EBSD, synthetic and an overlay; the default) or `'density'` (EBSD and synthetic on one shared colour scale, then their difference). Both populations get the same sample symmetry. Panel titles show (grains, plotted orientations).

In [ ]:
PFCMP_PLOT_MODE = 'scatter'            # or 'density'
PFCMP_UNIT_NORMALIZE = False           # density mode: compare pattern shape rather than strength

pretwin_pf = steps_visualization_export.pretwin_pole_figure_populations(rg, parent_info, base, assigner, CSL_LABEL)
fig, axes = steps_visualization_export.plot_pole_figure_comparison(
    pretwin_pf['pure_parents'], pretwin_pf['hosts'], ebsd_label='EBSD Pure Parents of Hosts',
    synth_label='Synthetic Host Grains', pole_family=POLE_FIGURE_FAMILY, plot_mode=PFCMP_PLOT_MODE,
    unit_normalize=PFCMP_UNIT_NORMALIZE, **PF_VIEW)
report.add_image(fig, title='Pole Figure Comparison -- EBSD vs Pre-Twin Synthetic', section='Orientation Assignment')
plt.show()

## Part L: Twin Generation

Adds primary twins (by default at host grain-boundary centroids), then secondary twins that grow inward or outward from the primary ones, aiming at the EBSD twin-area fraction measured in Part D. The structure is then cleaned topologically: spike voxels are removed and non-convex ("lobed") grains left by the voxel-carving step are split.

`vf_targets` (Part D) is passed in explicitly. Without it, primary twins would aim at the whole combined EBSD twin fraction on their own, and secondary twins would never be triggered. `TWIN_TVF_TOLERANCE` is how far past the target volume fraction introduction is allowed to overshoot before it stops. `TWIN_PROB_SEC_OUTWARD` / `TWIN_PROB_SEC_INWARD` set the split between 2a (outward, at the host interface) and 2b (inward, nested inside the primary) secondary twins.

In [ ]:
TWIN_N_LAMELLAE_PER_HOST = 10
TWIN_TVF_TOLERANCE = 0.05          # allowed overshoot past each stage's VF target before stopping
TWIN_TVF_2D_TO_3D_SCALE = 1.15     # fallback only -- unused once vf_targets (staged) is provided
TWIN_THICK_SCALE_FACTOR = 0.80     # scales the EBSD-measured thickness distribution used for lamellae
TWIN_MAX_VF_PER_HOST = 0.50        # cap on twin volume as a fraction of its own host grain's volume
TWIN_PROB_SEC_OUTWARD = 0.60       # secondary-twin nucleation: 2a (outward, at host interface)
TWIN_PROB_SEC_INWARD = 0.40        # secondary-twin nucleation: 2b (inward, nested in primary)
TWIN_RNG_SEED = 123

tg, tvf = steps_twin_generation.generate_twins(
    base, rg, parent_info, assigner, twin_thickness, vf_targets=vf_targets, csl_label=CSL_LABEL,
    n_lamellae_per_host=TWIN_N_LAMELLAE_PER_HOST, nucleation_site='gb_centroid', meshing_route='conformal',
    tvf_tolerance=TWIN_TVF_TOLERANCE, tvf_2d_to_3d_scale=TWIN_TVF_2D_TO_3D_SCALE,
    thick_scale_factor=TWIN_THICK_SCALE_FACTOR, max_vf_per_host=TWIN_MAX_VF_PER_HOST,
    prob_sec_outward=TWIN_PROB_SEC_OUTWARD, prob_sec_inward=TWIN_PROB_SEC_INWARD,
    rng_seed=TWIN_RNG_SEED)
tg.summary()

In [ ]:
cleaner, n_passes_run = steps_twin_generation.clean_structure(tg, n_passes=5)
len(cleaner.twin_role_clean), n_passes_run

## Part M: Post-Twin Validation

Checks the misorientation distribution of the cleaned, twinned structure against the full EBSD MDF (twins are present now, unlike in Pre-Twin Validation), per axis. It also compares the full 3D MDF curve directly against the EBSD curve from Part D.

`POSTTWIN_P_PERCENTAGE` / `POSTTWIN_WASSERSTEIN_THRESHOLD` are this stage's own pass-rate and distance thresholds. They're looser than the pre-twin ones by default, because slice-to-slice crystallographic variation is naturally wider than the variation in grain size checked before.

In [ ]:
POSTTWIN_N_SLICES = (10, 10, 10)
POSTTWIN_P_PERCENTAGE = 60.0             # required per-axis pass rate (%)
POSTTWIN_WASSERSTEIN_THRESHOLD = 5.0     # max Wasserstein distance for one slice to count as passing

post_validator, quat_3d_clean = steps_post_twin_validation.validate_crystallographic_representativeness(
    cleaner, mdf, n_slices=POSTTWIN_N_SLICES, test_axes=('x', 'y', 'z'),
    p_percentage=POSTTWIN_P_PERCENTAGE, wasserstein_threshold=POSTTWIN_WASSERSTEIN_THRESHOLD)
post_validator.overall_accepted

In [ ]:
mc_mdf_post = steps_post_twin_validation.compare_full_3d_mdf(cleaner, quat_3d_clean)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(mdf['hist_bin_centers'], mdf['hist_density'], label='EBSD')
ax.plot(mc_mdf_post['hist_bin_centers'], mc_mdf_post['hist_density'], label='SGC (post-twin)')
ax.set_xlabel('Misorientation angle (deg)'); ax.set_ylabel('Density'); ax.legend()
ax.set_title('Full 3D MDF: EBSD vs. Synthetic Grain Structure')
report.add_image(fig, title='Post-Twin Full 3D MDF', section='Post-Twin Validation')
plt.show()

### Per-Axis Pass Rate

The fraction of tested slices, per axis, that pass the Wasserstein-distance criterion at the threshold set above.

In [ ]:
ax_names = list(post_validator.axis_acceptance.keys())
pct_pass = [post_validator.axis_acceptance[a]['pct_pass'] for a in ax_names]
colors = ['seagreen' if post_validator.axis_acceptance[a]['accepted'] else 'crimson' for a in ax_names]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(ax_names, pct_pass, color=colors, alpha=0.75, edgecolor='black')
ax.axhline(60.0, color='steelblue', linestyle='--', linewidth=2, label='Pass threshold (60%)')
ax.set_ylabel('Slices passing (%)')
ax.set_title('Post-Twin crystallographic representativeness -- per-axis pass rate')
ax.set_ylim(0, 105)
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar, pct in zip(bars, pct_pass):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{pct:.0f}%',
            ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()

### Representative-Slice MDF versus EBSD

Plots the MDF of every slice that passed the criterion above (thin lines) with their per-axis mean (bold), against the full EBSD MDF. It shows what the misorientations in a representative 2D section look like.

In [ ]:
axes_present = [a for a in post_validator.slice_results if post_validator.slice_results[a]]
per_axis_mdf = steps_post_twin_validation.representative_slice_mdfs(
    post_validator, cleaner, quat_3d_clean, axes=axes_present)

fig, axs = plt.subplots(1, len(axes_present), figsize=(6 * len(axes_present), 4), squeeze=False)
for col, axis_name in enumerate(axes_present):
    ax = axs[0][col]
    ax.plot(mdf['hist_bin_centers'], mdf['hist_density'], 'k-', lw=2.0, label='EBSD full MDF')
    info = per_axis_mdf[axis_name]
    for slice_mdf in info['slice_mdfs']:
        ax.plot(slice_mdf['hist_bin_centers'], slice_mdf['hist_density'],
                color='#4878CF', alpha=0.25, lw=0.8)
    if info['mean_density'] is not None:
        ax.plot(info['mean_bin_centers'], info['mean_density'], color='#D65F5F', lw=2.0,
                label=f"Mean of {len(info['slice_mdfs'])} passing slice(s)")
    ax.set_title(f"{axis_name}-axis  ({info['n_passing']} passing slice(s))")
    ax.set_xlabel('Misorientation angle (deg)'); ax.set_ylabel('Probability density')
    ax.legend(fontsize=8)
fig.tight_layout()
report.add_image(fig, title='Representative-Slice MDF vs EBSD', section='Post-Twin Validation')
plt.show()

### Twin Thickness: EBSD Target versus Achieved 3D versus Apparent 2D

Compares three things: the EBSD target thickness distribution, the real 3D thickness put in during twin generation, and the apparent thickness you'd measure on a 2D cross-section (the minor-axis length of each twin's slice profile). It shows how much a 2D measurement over- or underestimates the true 3D lamella thickness.

In [ ]:
from scipy.stats import gaussian_kde

comparison = steps_post_twin_validation.twin_thickness_three_way_comparison(
    tg, cleaner, twin_thickness, post_validator)
ebsd_um, ebsd_mean_um = comparison['ebsd_um'], comparison['ebsd_mean_um']
actual_3d_um, actual_3d_mean_um = comparison['actual_3d_um'], comparison['actual_3d_mean_um']
per_axis_um, per_axis_means_um = comparison['per_axis_um'], comparison['per_axis_means_um']

all_maxes = [a.max() for a in (ebsd_um, actual_3d_um, *per_axis_um.values()) if a.size]
grid = np.linspace(0, (max(all_maxes) * 1.1) if all_maxes else 1.0, 300)
axis_colors = {'X': 'tomato', 'Y': 'steelblue', 'Z': 'seagreen'}

fig, ax = plt.subplots(figsize=(9, 5))
if ebsd_um.size > 1:
    ax.plot(grid, gaussian_kde(ebsd_um)(grid), 'k--', lw=2,
            label=f'EBSD target  mean={ebsd_mean_um:.2f} um')
if actual_3d_um.size > 1:
    ax.plot(grid, gaussian_kde(actual_3d_um)(grid), color='mediumpurple', lw=2.5,
            label=f'Actual 3D introduced  n={actual_3d_um.size}  mean={actual_3d_mean_um:.2f} um')
for axis_name, values in per_axis_um.items():
    if values.size > 1:
        ax.plot(grid, gaussian_kde(values)(grid), color=axis_colors.get(axis_name, 'gray'), lw=2.2,
                label=f'{axis_name}-normal slices  n={values.size}  mean={per_axis_means_um[axis_name]:.2f} um')
ax.set_xlabel('Twin lamella thickness (um)'); ax.set_ylabel('Density')
ax.set_title('Twin thickness: EBSD target | actual 3D | 2D-slice apparent')
ax.legend(fontsize=8.5); ax.grid(alpha=0.3)
fig.tight_layout()
report.add_image(fig, title='Twin Thickness Three-Way Comparison', section='Post-Twin Validation')
plt.show()

### Pole Figure Comparison: EBSD versus Post-Twin Synthetic

The full EBSD map (twins counted as their own grains) against every grain of the cleaned synthetic structure. Settings are the same as in the pre-twin comparison above.

In [ ]:
posttwin_ebsd = steps_ebsd_analysis_2.ebsd_pole_figure_stages(rg, parent_info, csl_label=CSL_LABEL)['full']
posttwin_sgs = steps_visualization_export.sgs_pole_figure_stages(cleaner)['full']
fig, axes = steps_visualization_export.plot_pole_figure_comparison(
    posttwin_ebsd, posttwin_sgs, ebsd_label='EBSD Full', synth_label='Synthetic All Grains',
    pole_family=POLE_FIGURE_FAMILY, plot_mode=PFCMP_PLOT_MODE, unit_normalize=PFCMP_UNIT_NORMALIZE, **PF_VIEW)
report.add_image(fig, title='Pole Figure Comparison -- EBSD vs Post-Twin Synthetic', section='Post-Twin Validation')
plt.show()

## Part N: Distribution Viewer

Compares the finished synthetic structure (SGS) with EBSD on four properties: misorientation, twin lamella thickness, host grain size and twin volume fraction, pooled over the sampled slices. This is the main test of how well the SGS agrees with the reference EBSD data.

In [ ]:
cmp_data = steps_distribution_viewer.compare_ebsd_vs_sgs(
    cleaner, tg, base, rg, parent_info, mdf, twin_thickness, n_per_axis=5)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (prop_name, vals) in zip(axes.ravel(), cmp_data.items()):
    ax.hist(vals['ebsd'], bins=25, alpha=0.5, density=True, label='EBSD')
    ax.hist(vals['sgs'], bins=25, alpha=0.5, density=True, label='SGS')
    ax.set_title(steps_distribution_viewer.COMPARISON_PROP_LABELS[prop_name])
    ax.legend()
fig.tight_layout()
report.add_image(fig, title='EBSD vs. SGS Comparison Properties', section='Distribution Viewer')
plt.show()

### Grain-Role Property Comparison

The comparison above pools all grains. This one splits morphology by role instead: non-hosting matrix grains, host grains and twin grains (any generation), EBSD versus the synthetic structure (SGS).

The two role schemes don't map one-to-one. EBSD uses pure_parents / pure_twins / intermediates / non_role, the SGS uses nonhost / host / primary / seca / secb. An EBSD "intermediate" (a twin that itself hosts another twin) has no direct SGS equivalent, since primary versus secondary in the SGS is about which nucleation generation a twin belongs to, not whether it hosts another twin. So both schemes are reduced to three categories that can be compared sensibly: non-hosting, host, and twin (any generation, on either side).

In [ ]:
role_prop_data = steps_distribution_viewer.compare_role_properties(
    rg, parent_info, cleaner, voxel_size=base.voxel_size,
    selected_props=('area', 'aspect_ratio', 'perimeter', 'solidity', 'n_neighbours'))

buckets = ['non_hosting', 'host', 'twin']
props_shown = list(role_prop_data.keys())
fig, axes = plt.subplots(len(props_shown), len(buckets),
                          figsize=(4 * len(buckets), 3.5 * len(props_shown)), squeeze=False)
for row, prop in enumerate(props_shown):
    for col, bucket in enumerate(buckets):
        ax = axes[row][col]
        vals = role_prop_data[prop][bucket]
        if vals['ebsd'].size:
            ax.hist(vals['ebsd'], bins=20, alpha=0.5, density=True, label='EBSD')
        if vals['sgs'].size:
            ax.hist(vals['sgs'], bins=20, alpha=0.5, density=True, label='SGS')
        ax.set_title(f'{prop} -- {bucket}')
        if row == 0 and col == 0:
            ax.legend()
fig.tight_layout()
report.add_image(fig, title='Grain-Role Property Comparison (EBSD vs SGS)', section='Distribution Viewer')
plt.show()

## Part O: Subsetting (Optional)

Off by default. Cuts one or more cuboid sub-volumes out of the fully twinned, cleaned 3D structure, for when you need a representative volume element smaller than the whole domain. With the defaults below (one tile covering 20% of the domain from the origin) it does effectively nothing, and Visualization and Export works fine on `cleaner` without it.

In [ ]:
subset_cleaners = steps_subsetting.generate_subsets(
    cleaner, start_pct=(0.0, 0.0, 0.0), n_cuboids=(1, 1, 1),
    length_pct=(20.0, 20.0, 20.0), stride_pct=(20.0, 20.0, 20.0), overflow_policy='clip')
len(subset_cleaners)

## Part P: Visualization and Export

Renders the finished 3D structure interactively, writes the cleaned structure's raw arrays (`.npy` / `.pkl`), and exports a full Abaqus voxel-conformal mesh (`.inp` files) with element sets by role, family and variant, and node sets on each domain face.

In [ ]:
# Interactive PyVista render -- opens a separate window (or renders inline
# with a Jupyter PyVista backend enabled). Skip this cell in a headless run.
steps_visualization_export.render_3d_structure(cleaner)

### IPF-Coloured 2D Slice

An alternative to the interactive 3D render above: one 2D cross-section coloured by inverse-pole-figure (IPF) orientation. It uses plain matplotlib and shows inline in any environment.

In [ ]:
fig, ax = steps_visualization_export.render_ipf_slice(cleaner, axis=2)
report.add_image(fig, title='IPF-Coloured Z-Normal Slice', section='Visualization & Export')

### Pole Figures

The same four populations as the EBSD pole figures in Part D, now for the final synthetic structure: full structure, hosts only (the SGS counterpart of EBSD "parents": every non-twin grain, whether or not it was allocated as a host), twins only, and hosts and twins overlaid with different markers. All pole-figure settings (family, plot mode, projection, hemisphere, sample symmetry) keep the values from Part D.

In [ ]:
sgs_pf_stages = steps_visualization_export.sgs_pole_figure_stages(cleaner)

fig, ax = steps_visualization_export.plot_pole_figure(
    sgs_pf_stages['full'], pole_family=POLE_FIGURE_FAMILY, plot_mode=POLE_FIGURE_PLOT_MODE,
    half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG, **PF_VIEW, title='SGS Full Structure')
report.add_image(fig, title='SGS Pole Figure -- Full Structure', section='Visualization & Export')
plt.show()

fig, ax = steps_visualization_export.plot_pole_figure(
    sgs_pf_stages['host'], pole_family=POLE_FIGURE_FAMILY, plot_mode=POLE_FIGURE_PLOT_MODE,
    half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG, **PF_VIEW, title='SGS Hosts Only')
report.add_image(fig, title='SGS Pole Figure -- Hosts Only', section='Visualization & Export')
plt.show()

fig, ax = steps_visualization_export.plot_pole_figure(
    sgs_pf_stages['all_twins'], pole_family=POLE_FIGURE_FAMILY, plot_mode=POLE_FIGURE_PLOT_MODE,
    half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG, **PF_VIEW, title='SGS Twins Only')
report.add_image(fig, title='SGS Pole Figure -- Twins Only', section='Visualization & Export')
plt.show()

fig, ax = steps_visualization_export.plot_pole_figure_overlay(
    sgs_pf_stages['host'], 'Hosts', sgs_pf_stages['all_twins'], 'Twins',
    pole_family=POLE_FIGURE_FAMILY, **PF_OVERLAY_VIEW, title='SGS Hosts vs. Twins')
report.add_image(fig, title='SGS Pole Figure -- Hosts vs Twins Overlay', section='Visualization & Export')
plt.show()

In [ ]:
raw_result = steps_visualization_export.export_raw(cleaner)
raw_result

In [ ]:
index = steps_visualization_export.build_abaqus_index(cleaner)

In [ ]:
abq_result = steps_visualization_export.export_abaqus_mesh(
    cleaner, tg, index, base_filename='twinned_fcc_mesh', voxel_size_um=1.0,
    element_type='C3D8', material_format='bunge_euler', n_depvar=100,
    material_level='feature')
abq_result

In [ ]:
abq_summary = {k: v for k, v in abq_result.items() if k != 'files'}
report.add_table([abq_summary], title='Abaqus Mesh Export', section='Visualization & Export')

## Summary: Achieved versus Target Total Twin Volume Fraction

The fraction of the final, cleaned structure's volume that is twin material (primary and secondary together), against the full EBSD-partitioned target (Stage-1 + Secondary-2a + Secondary-2b from `vf_targets`, Part D). It's evaluated on `cleaner`, not `tg`, because cleaning (spike removal, lobe splitting) changes voxel counts after twins are added.

Don't use `tg.summary()['tvf_achieved_pct_of_target']`, reported above, for this. It compares the pre-cleaning, all-twins volume against `tvf_target_3d`, the Stage-1-only target that `introduce_primary_twins` uses as its stopping threshold, not the combined total. Once secondary twins are in, it therefore reads higher than the true achieved fraction.

In [ ]:
twin_gids = {g for g, role in cleaner.twin_role_clean.items()
             if role in ('primary_twin', 'secondary_twin')}
twin_vox = int(np.isin(cleaner.lgi_clean, list(twin_gids)).sum())
total_vox = int(np.sum(cleaner.lgi_clean > 0))
achieved_total_vf = twin_vox / total_vox if total_vox > 0 else 0.0

target_total_vf = (vf_targets['tvf_stage1'] + vf_targets['tvf_secondary_2a']
                    + vf_targets['tvf_secondary_2b'])
pct_of_target = 100 * achieved_total_vf / target_total_vf if target_total_vf > 0 else 0.0

print(f'Target total twin VF (EBSD-partitioned, 3D) : {target_total_vf:.4f}')
print(f'Achieved total twin VF (cleaned structure)   : {achieved_total_vf:.4f}')
print(f'Achieved / Target                            : {pct_of_target:.1f}%')

report.add_table(
    [{'target_total_vf': target_total_vf, 'achieved_total_vf': achieved_total_vf,
      'pct_of_target': pct_of_target}],
    title='Achieved vs. Target Total Twin Volume Fraction', section='Twin Generation')

## Texture and Misorientation Representativeness

The earlier checks (Pre-Twin Validation, Post-Twin Validation, twin volume fraction) tell you whether the structure meets thresholds you set. The three metrics below are different: they're continuous, unthresholded numbers for two things those checks don't cover directly. One is crystallographic texture (the distribution of absolute grain orientations, as opposed to neighbour-pair misorientation). The other is an overall neighbour-misorientation agreement score. There's no pass/fail for any of the three, since the rest of the pipeline has no threshold for them. The values are just reported so you can judge them yourself.

In [ ]:
REPR_MATCH_TOLERANCE_PCT = 25.0   # acceptance band (+/- %) applied to morphology/thickness mean ratios below

mdf_repr = steps_post_twin_validation.mdf_representativeness_score(mdf, mc_mdf_post)
mdf_repr_score = mdf_repr['representativeness_score'] if mdf_repr else None

twin_thick_repr = steps_post_twin_validation.twin_thickness_representativeness_score(comparison)
twin_thick_repr_score = twin_thick_repr['representativeness_score'] if twin_thick_repr else None
twin_thick_repr_ratio = twin_thick_repr['ratio'] if twin_thick_repr else None
twin_thick_accepted = (twin_thick_repr_ratio is not None
                        and abs(twin_thick_repr_ratio - 1.0) <= REPR_MATCH_TOLERANCE_PCT / 100.0)

texture_vf_comparison = steps_distribution_viewer.compute_texture_component_vf_comparison(rg, cleaner)
pf_delta_mud = steps_distribution_viewer.compute_pole_figure_delta_mud_iqr(
    rg, cleaner, pole_family=POLE_FIGURE_FAMILY, apply_sample_symmetry=PF_APPLY_SAMPLE_SYMMETRY, use_rd=PF_USE_RD, use_td=PF_USE_TD, use_nd=PF_USE_ND)
role_prop_summary = steps_distribution_viewer.role_property_match_summary(
    role_prop_data, tolerance_pct=REPR_MATCH_TOLERANCE_PCT)

print('Neighbour misorientation (MDF) representativeness score  :',
      f'{mdf_repr_score:.3f}  (0-1, higher = more alike)' if mdf_repr_score is not None else 'N/A')
print('Twin thickness (EBSD target vs. achieved 3D)              :',
      (f'ratio={twin_thick_repr_ratio:.2f}  score={twin_thick_repr_score:.3f}  '
       f"{'ACCEPTED' if twin_thick_accepted else 'outside tolerance'}")
      if twin_thick_repr_ratio is not None else 'N/A')
print('Texture component VF total deviation                      :',
      f"{texture_vf_comparison['total_abs_deviation_pct']:.1f} pp across "
      f"{len(texture_vf_comparison['matched'])} matched components")
print(f'Pole figure {{{POLE_FIGURE_FAMILY}}} delta-MUD IQR                            :',
      f"{pf_delta_mud['iqr']:.3f}  (lower = more alike)")
print(f'Grain-role morphology match (tolerance +/-{REPR_MATCH_TOLERANCE_PCT:.0f}%)             :')
for prop, s in role_prop_summary.items():
    tag = 'ACCEPTED' if s['accepted'] else 'outside tolerance'
    ratio_str = f"ratio={s['ratio']:.2f}" if s['ratio'] is not None else 'ratio=N/A'
    score_str = f"score={s['score']:.3f}" if s['score'] is not None else 'score=N/A'
    print(f'    {prop:14s} {ratio_str:14s} {score_str:14s} {tag}')

report.add_table(
    [{'component': m['component'], 'ebsd_vf_pct': m['ebsd_vf_pct'],
      'sgs_vf_pct': m['sgs_vf_pct'], 'abs_diff_pct': m['abs_diff_pct']}
     for m in texture_vf_comparison['matched']] or [{'component': 'none matched'}],
    title='Texture Component Volume-Fraction Comparison', section='Distribution Viewer')

### Texture Residual Pole Figures

The three-panel EBSD | SGS | Difference pole-figure plot used elsewhere in this pipeline to compare two orientation populations. The first set (full structure versus full structure) reproduces the delta-MUD interquartile range reported above exactly, because it's computed from the same populations. The second (parents versus hosts) adds a twin-role-specific comparison rather than a pooled one.

In [ ]:
fig, axes, iqr_full = steps_distribution_viewer.plot_texture_residual(
    ebsd_pf_stages['full'], sgs_pf_stages['full'],
    pole_family=POLE_FIGURE_FAMILY, half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG,
    apply_sample_symmetry=PF_APPLY_SAMPLE_SYMMETRY, use_rd=PF_USE_RD, use_td=PF_USE_TD, use_nd=PF_USE_ND,
    title_ebsd='EBSD Full', title_sgs='SGS Full')
report.add_image(fig, title='Texture Residual -- Full Structure', section='Distribution Viewer')
plt.show()
print(f'Full-structure delta-MUD IQR: {iqr_full:.3f}')

fig, axes, iqr_parents = steps_distribution_viewer.plot_texture_residual(
    ebsd_pf_stages['parents'], sgs_pf_stages['host'],
    pole_family=POLE_FIGURE_FAMILY, half_width_deg=POLE_FIGURE_HALF_WIDTH_DEG,
    apply_sample_symmetry=PF_APPLY_SAMPLE_SYMMETRY, use_rd=PF_USE_RD, use_td=PF_USE_TD, use_nd=PF_USE_ND,
    title_ebsd='EBSD Parents', title_sgs='SGS Hosts')
report.add_image(fig, title='Texture Residual -- Parents vs Hosts', section='Distribution Viewer')
plt.show()
print(f'Parents-vs-hosts delta-MUD IQR: {iqr_parents:.3f}')

## Results Scorecard

The pass/fail and percentage results so far are spread across Parts J and M and the volume-fraction summary above. This section pulls them into one overall assessment of how the pipeline did.

In [ ]:
pre_axes = pre_validator.axis_acceptance
post_axes = post_validator.axis_acceptance
twin_vf_ok = 90.0 <= pct_of_target <= 110.0
overall_verdict = ('PASS' if pre_validator.overall_accepted
                    and post_validator.overall_accepted and twin_vf_ok
                    else 'REVIEW NEEDED')

n_morph_accepted = sum(1 for s in role_prop_summary.values() if s['accepted'])
n_morph_total = len(role_prop_summary)

scorecard_rows = [
    {'stage': 'Pre-Twin Validation',
     'result': 'PASS' if pre_validator.overall_accepted else 'FAIL',
     'detail': f"{sum(a['accepted'] for a in pre_axes.values())}/{len(pre_axes)} axes accepted"},
    {'stage': 'Post-Twin Validation',
     'result': 'PASS' if post_validator.overall_accepted else 'FAIL',
     'detail': f"{sum(a['accepted'] for a in post_axes.values())}/{len(post_axes)} axes accepted"},
    {'stage': 'Twin Volume Fraction',
     'result': 'PASS' if twin_vf_ok else 'REVIEW',
     'detail': f'{pct_of_target:.1f}% of target ({achieved_total_vf:.4f} vs {target_total_vf:.4f})'},
    {'stage': 'Neighbour Misorientation (MDF) Repr.',
     'result': 'INFO',
     'detail': (f'score={mdf_repr_score:.3f} (0-1, higher=more alike)'
                if mdf_repr_score is not None else 'N/A')},
    {'stage': 'Twin Thickness (EBSD target vs. achieved 3D)',
     'result': 'PASS' if twin_thick_accepted else 'REVIEW',
     'detail': (f'ratio={twin_thick_repr_ratio:.2f} ({(twin_thick_repr_ratio - 1) * 100:+.1f}% vs EBSD), '
                f'score={twin_thick_repr_score:.3f}'
                if twin_thick_repr_ratio is not None else 'N/A')},
    {'stage': 'Texture -- Component VF Match',
     'result': 'INFO',
     'detail': (f"{texture_vf_comparison['total_abs_deviation_pct']:.1f} pp total deviation across "
                f"{len(texture_vf_comparison['matched'])} matched components")},
    {'stage': 'Texture -- Pole Figure Delta-MUD IQR',
     'result': 'INFO',
     'detail': f"IQR={pf_delta_mud['iqr']:.3f} (lower=more alike)"},
]
for prop, s in role_prop_summary.items():
    scorecard_rows.append({
        'stage': f'Morphology -- {prop}',
        'result': 'PASS' if s['accepted'] else 'REVIEW',
        'detail': (f"ratio={s['ratio']:.2f} ({(s['ratio'] - 1) * 100:+.1f}% vs EBSD), score={s['score']:.3f}"
                   if s['ratio'] is not None else 'N/A'),
    })
scorecard_rows.append({
    'stage': 'Morphology -- Overall',
    'result': 'PASS' if n_morph_accepted == n_morph_total else 'REVIEW',
    'detail': f'{n_morph_accepted}/{n_morph_total} properties within +/-{REPR_MATCH_TOLERANCE_PCT:.0f}% of EBSD',
})

print(f'Overall verdict: {overall_verdict}')
print()
for row in scorecard_rows:
    print(f"  [{row['result']:>6}]  {row['stage']:<44} {row['detail']}")

report.add_table(scorecard_rows, title='Results Scorecard', section='Summary')

## Temporal-Slice Sweep

Part G picks one "best" temporal slice from cheap 2D cross-section comparisons alone. It can't know a slice's actual total twin volume fraction, because that only exists after the whole downstream pipeline (host allocation, orientation assignment, twin generation, cleaning) has run on that slice. This section runs that chain for every slice you ask for and reports which ones reach `SWEEP_VF_THRESHOLD_PCT` of the EBSD-partitioned target.

`SWEEP_TSLICE_KEYS`: `None` (or an empty list) sweeps every saved slice except the un-annealed seed at index 0. An explicit list restricts the sweep to those keys; keys that aren't in this run are dropped and reported, not raised as errors. Host allocation and twin generation reuse the settings from Parts I and L. Orientation assignment is a single fast `conflict_free` run per slice, rather than repeating Part K's optimization for every slice, which would multiply the cost of this section by roughly ten. The volume fraction you get depends mainly on host geometry and the twin-introduction probabilities, not on the assignment mode.

`SWEEP_N_WORKERS`: `1` (default) sweeps the slices one after another in this process. Above 1, the slices are spread over that many worker processes, since each slice's chain is independent. It uses processes, not threads, because the per-slice work is ordinary CPU/NumPy-bound Python that the interpreter's global lock would stop from overlapping across threads. The inputs shared by all slices (`pxt`, `rg`, `parent_info` and the four kwargs dicts) are sent to each worker once at start-up, not once per slice. It only helps when more than one slice is swept, and if you have fewer cores or fewer slices than `SWEEP_N_WORKERS`, the smaller number wins.

Exports: every slice that reaches the threshold is written under ONE shared master folder for the sweep, `<output_dir>/TwinnedFCC/Grain Structures/<base_filename><N>/` (auto-numbered once at the start of the sweep, not per slice), each in its own `tslice_<key>/` subfolder. A single export, such as `export_raw()` in Part P, writes straight into its own master folder with no `tslice_<key>` level. Both follow the same convention and never collide, within a run or across repeated runs. The sweep doesn't export Abaqus meshes. If you need one for a particular passing slice, re-run Parts I-P for that `tslice_key`.

In [ ]:
SWEEP_TSLICE_KEYS = None            # None/[] = every saved slice except index 0; or an explicit list of keys
SWEEP_VF_THRESHOLD_PCT = 90.0       # minimum achieved-VF-as-percent-of-target to count as passing
SWEEP_N_WORKERS = 3                 # 1 = sequential; >1 = that many worker processes (see markdown above)

sweep_rows = steps_temporal_slice_sweep.sweep_temporal_slices(
    pxt, rg, parent_info, vf_targets, twin_thickness,
    tslice_keys=SWEEP_TSLICE_KEYS, vf_threshold_pct=SWEEP_VF_THRESHOLD_PCT,
    n_workers=SWEEP_N_WORKERS,
    host_alloc_kwargs=dict(
        target_fraction=HOST_TARGET_FRACTION, min_voxels=HOST_MIN_VOXELS,
        scale_factor=HOST_2D_TO_3D_SCALE_FACTOR, ranking_weight=HOST_RANKING_VOLUME_WEIGHT,
        mis_fraction=HOST_MIS_FRACTION, mis_runs=HOST_MIS_RUNS, seed=HOST_SEED,
        pool_b_band_shape=HOST_POOL_B_BAND_SHAPE, pool_b_n_std=HOST_POOL_B_N_STD),
    orientation_kwargs=dict(csl_label=CSL_LABEL, ori_mode='conflict_free', rng_seed=42),
    twin_gen_kwargs=dict(
        csl_label=CSL_LABEL, n_lamellae_per_host=TWIN_N_LAMELLAE_PER_HOST,
        nucleation_site='gb_centroid', meshing_route='conformal',
        tvf_tolerance=TWIN_TVF_TOLERANCE, thick_scale_factor=TWIN_THICK_SCALE_FACTOR,
        max_vf_per_host=TWIN_MAX_VF_PER_HOST, prob_sec_outward=TWIN_PROB_SEC_OUTWARD,
        prob_sec_inward=TWIN_PROB_SEC_INWARD, rng_seed=TWIN_RNG_SEED),
    clean_kwargs=dict(n_passes=5))

n_passing = sum(1 for r in sweep_rows if r.get('passes'))
print()
print(f'{n_passing}/{len(sweep_rows)} swept slice(s) reached {SWEEP_VF_THRESHOLD_PCT:.0f}% of target.')

sweep_table_rows = [{k: v for k, v in r.items() if k != 'export'} for r in sweep_rows]
report.add_table(sweep_table_rows, title='Temporal-Slice Sweep', section='Summary')

## Finish: Report Generation

Writes the accumulated `add_table` / `add_image` entries into a single `report.html`.

In [ ]:
report_path = steps_visualization_export.finish_report(report)
print('Report written to:', report_path)